# Spotify Genre Classification

## Ingeniería de Datos - Inspección inicial del dataset

**Pregunta de investigación:**  
¿Es posible predecir el género musical de una canción utilizando únicamente sus características de audio?

Este notebook presenta una inspección sencilla de los datos obtenidos mediante el pipeline ETL del proyecto.

El pipeline completo se encuentra implementado en los scripts de la carpeta `src/`.

Para facilitar la visualización, en este notebook trabajaremos con una muestra reproducible de 500 registros.

## 1. Librerías utilizadas

Para trabajar con los datos utilizamos principalmente:

- **pandas**: permite cargar, manipular y analizar datos tabulares.
- **pathlib**: permite trabajar con rutas de archivos de manera independiente del sistema operativo.

In [ ]:
from pathlib import Path
import pandas as pd

: 

## 2. Carga del dataset

El pipeline genera un dataset procesado a partir de la fuente original.

Para esta inspección utilizaremos una muestra de 500 registros con el objetivo de visualizar los datos de forma simple y rápida. La semilla `random_state=42` permite obtener la misma muestra en cada ejecución.

In [ ]:
# La ruta funciona al ejecutar el notebook desde la raíz del proyecto o desde notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

processed_path = PROJECT_ROOT / "data" / "processed" / "spotify_audio_features.csv"
raw_path = PROJECT_ROOT / "data" / "raw" / "spotify_tracks.csv"

if not processed_path.exists() or not raw_path.exists():
    raise FileNotFoundError(
        "No se encontraron los datos generados. Ejecutá primero: python src/pipeline.py"
    )

df = pd.read_csv(processed_path)
df_sample = df.sample(n=500, random_state=42)

In [ ]:
df_sample.head()

Cada fila representa un registro de canción asociado a sus características de audio y a un género musical.

In [ ]:
filas, columnas = df_sample.shape
print(f"La muestra contiene {filas} observaciones y {columnas} columnas.")
df_sample.shape

El resultado de `shape` se interpreta como `(filas, columnas)`. Cada observación representa un registro de canción.

In [ ]:
df_sample.columns.tolist()

Las variables corresponden principalmente a características de audio de Spotify. `track_genre` representa la variable objetivo que posteriormente intentaremos predecir.

## 3. Información del dataset

Esta información permite conocer la cantidad de registros, las columnas, sus tipos de datos y la presencia de valores no nulos.

In [ ]:
df_sample.info()

In [ ]:
df_sample.dtypes.to_frame(name="tipo_de_dato")

## 4. Géneros musicales

`track_genre` será posteriormente la variable objetivo del problema de clasificación.

In [ ]:
cantidad_generos = df_sample["track_genre"].nunique()
print(f"Géneros presentes en la muestra: {cantidad_generos}")

In [ ]:
sorted(df_sample["track_genre"].dropna().unique())

In [ ]:
df_sample["track_genre"].value_counts()

## 5. Canciones únicas y registros repetidos

El dataset puede contener más de un registro asociado a la misma canción. Por eso, la cantidad de filas no necesariamente coincide con la cantidad de canciones únicas.

Para este análisis utilizamos el dataset raw completo, ya que conserva `track_id`, `track_name` y `artists`.

In [ ]:
df_raw = pd.read_csv(raw_path)

total_registros = len(df_raw)
canciones_unicas = df_raw["track_id"].nunique()
registros_repetidos = total_registros - canciones_unicas

resumen_canciones = pd.Series({
    "Total de registros": total_registros,
    "Canciones únicas por track_id": canciones_unicas,
    "Registros adicionales con track_id repetido": registros_repetidos,
})

resumen_canciones.to_frame(name="cantidad")

Una misma canción puede aparecer más de una vez en el dataset. Esto puede ocurrir, por ejemplo, si está asociada a diferentes categorías o géneros.

Por este motivo, antes de eliminar duplicados automáticamente debemos analizar qué representan estos registros.

In [ ]:
duplicated_tracks = df_raw[
    df_raw.duplicated(subset="track_id", keep=False)
].sort_values("track_id")

duplicated_tracks[
    ["track_id", "artists", "track_name", "track_genre"]
].head(20)

In [ ]:
nombres_unicos = df_raw["track_name"].nunique()
print(f"Nombres de canciones únicos: {nombres_unicos}")

`track_name` no es un identificador confiable por sí solo, porque distintas canciones pueden compartir el mismo nombre. Para contar canciones únicas se utiliza preferentemente `track_id`.

## 6. Resumen inicial

A partir de esta inspección podemos observar:

- el dataset contiene características numéricas relacionadas con el audio de cada canción;
- `track_genre` será la variable objetivo del problema de clasificación;
- existen múltiples géneros musicales;
- la cantidad de registros no necesariamente coincide con la cantidad de canciones únicas;
- existen canciones que aparecen repetidas en el dataset.

Estas observaciones serán tenidas en cuenta posteriormente durante las etapas de limpieza, análisis exploratorio y modelado.